# 04 — Hand Detection: Bounding Box, Confidence, IoU

## 1. Learning Goal

After this notebook you can explain, **and show with code**:

> A detector looks at an image and answers: *"Is there a hand? Where? How sure are you?"*

You will learn:

1. What a **detection** is: a **bounding box** + a **confidence score**
2. How a box differs from **keypoints** (preview of notebook 05)
3. How to compute a box from keypoints, and **crop** the hand (uses notebook 01!)
4. **IoU** (Intersection over Union): the standard way to measure how good a box is
5. **True positive, false positive, false negative**, and how the confidence threshold changes them

## 2. Concept

```text
Image ──► Model ──► Bounding box (x1, y1, x2, y2)
                    + Confidence (0.0 … 1.0)
```

```text
(x1,y1) ┌─────────────┐
        │   ✋        │      4 numbers describe the box.
        │             │      1 number says how sure the model is.
        └─────────────┘ (x2,y2)
```

## 3. Why do we need this?

Our final system starts with the webcam. Before we can recognise a sign, we must know **if there is a hand and where it is**. The box lets us crop the hand and ignore the background.

> Note: the pretrained MediaPipe model used here finds the hand and its 21 keypoints together. We will derive the box from the keypoints. In notebook 05 you will also train a YOLO model that predicts hand keypoints.

## 0. Setup

You need `pip install mediapipe opencv-python numpy matplotlib jupyter`, the model file `models/hand_landmarker.task`, and a photo of your hand at `data/myhand.jpg` (the one from Phase 1 Step 2).

In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Works whether you start Jupyter in the project folder or inside notebooks/
ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA = ROOT / "data"
MODELS = ROOT / "models"
DATA.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

def show_bgr(img, title="", size=6):
    plt.figure(figsize=(size, size))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # OpenCV is BGR, Matplotlib wants RGB
    plt.title(title)
    plt.axis("off")
    plt.show()

def describe(image, name="image"):
    print(f"--- {name} ---")
    print("shape :", image.shape, "| dtype:", image.dtype, "| min:", image.min(), "| max:", image.max())

---
# Part 1 — What does a detector actually produce?

## 4. Visual intuition + 6. Implementation

The helper functions below wrap MediaPipe. Read the docstrings. The important part is the **last function**: it turns the result object into a plain NumPy array.

In [ ]:
import mediapipe as mp

def create_detector(running_mode="image", num_hands=1, min_detection=0.5):
    """Load the MediaPipe Hand Landmarker model file from models/."""
    modes = {"image": mp.tasks.vision.RunningMode.IMAGE,
             "video": mp.tasks.vision.RunningMode.VIDEO}
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODELS / "hand_landmarker.task")),
        running_mode=modes[running_mode],
        num_hands=num_hands,
        min_hand_detection_confidence=min_detection,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)

def detect_hand(detector, bgr_image):
    """OpenCV gives BGR, MediaPipe wants RGB. Convert, wrap, detect."""
    rgb = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    return detector.detect(mp_image)

def landmarks_to_array(result, hand_index=0):
    """result -> NumPy array of shape (21, 3): columns are x, y, z (x and y are fractions 0..1)."""
    hand = result.hand_landmarks[hand_index]
    return np.array([[lm.x, lm.y, lm.z] for lm in hand], dtype=np.float32)

In [ ]:
PHOTO = DATA / "myhand.jpg"          # change this if your file has another name

img = cv2.imread(str(PHOTO))
print("image loaded?", img is not None)
describe(img, "my hand photo")
H, W = img.shape[:2]
show_bgr(img, "My photo")

In [ ]:
detector = create_detector()                      # loads models/hand_landmarker.task
result = detect_hand(detector, img)

print("hands found      :", len(result.hand_landmarks))
print("fields in result :", [name for name in dir(result) if not name.startswith("_")])
print("left or right    :", result.handedness[0][0].category_name,
      "| score:", round(result.handedness[0][0].score, 3))

lm = landmarks_to_array(result)                   # shape (21, 3)
describe(lm, "landmarks (fractions)")

### What to observe

- `hands found: 1` means one detection. With no hand, this would be `0` and the lists are empty.
- The result has **landmarks** (21 points), **world landmarks** (same points in metres) and **handedness** (left/right + score).
- `lm` has shape `(21, 3)`: 21 points × `(x, y, z)`. `x` and `y` are fractions of the image width/height.

## Convert fractions to pixels

```text
pixel_x = x * width        pixel_y = y * height
```

In [ ]:
points_px = (lm[:, :2] * np.array([W, H])).astype(int)    # shape (21, 2), columns (x, y)
print("wrist pixel     :", points_px[0])
print("index tip pixel :", points_px[8])

---
# Part 2 — From keypoints to a bounding box

## Concept

A box is the **smallest rectangle that contains all 21 points** (plus some padding so we do not cut off fingers):

```text
x1 = min of all x  - pad          y1 = min of all y  - pad
x2 = max of all x  + pad          y2 = max of all y  + pad
```

In [ ]:
def box_from_keypoints(points_px, pad=20, img_w=None, img_h=None):
    x1, y1 = points_px.min(axis=0) - pad
    x2, y2 = points_px.max(axis=0) + pad
    if img_w is not None:                      # keep the box inside the image
        x1, x2 = max(0, x1), min(img_w - 1, x2)
    if img_h is not None:
        y1, y2 = max(0, y1), min(img_h - 1, y2)
    return int(x1), int(y1), int(x2), int(y2)

box = box_from_keypoints(points_px, pad=20, img_w=W, img_h=H)
print("box (x1, y1, x2, y2):", box)

x1, y1, x2, y2 = box
annotated = img.copy()
cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 3)

crop = img[y1:y2, x1:x2]                       # rows first (y), then columns (x)!
describe(crop, "hand crop")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); axes[0].set_title("Box from keypoints")
axes[1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB));      axes[1].set_title("Cropped hand")
for ax in axes:
    ax.axis("off")
plt.show()

### What to observe

- The crop is exactly what notebook 01 called "slicing": `img[y1:y2, x1:x2]`.
- A box says **where** the hand is. It does **not** say whether the fingers are open or closed. For sign language, that shape is what matters, which is why keypoints come next.

### Challenge 1

Change `pad` to `0`, `20`, `80`. What happens to the crop? Which value would you pick if you wanted to feed the crop to a CNN later, and why?

---
# Part 3 — IoU: how do we measure a good box?

## Concept

Suppose a human drew the **true** box and the model predicted another one. How close are they?

```text
IoU = Area of overlap / Area of union
```

```text
 ┌───────────┐
 │   A   ┌───┼───────┐        overlap  = the region in both boxes
 │       │ ∩ │       │        union    = everything covered by A or B
 └───────┼───┘   B   │        IoU = 0   → no overlap at all
         └───────────┘        IoU = 1   → identical boxes
```

Common rule: a detection counts as **correct** if IoU ≥ 0.5.

## Calculate one by hand first

A = (0, 0, 100, 100) and B = (50, 50, 150, 150). Try it on paper before running the next cell.

In [ ]:
A = (0, 0, 100, 100)
B = (50, 50, 150, 150)

# Overlap rectangle: the right-most left edge, the bottom-most top edge, etc.
ix1, iy1 = max(A[0], B[0]), max(A[1], B[1])
ix2, iy2 = min(A[2], B[2]), min(A[3], B[3])
inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)

area_A = (A[2] - A[0]) * (A[3] - A[1])
area_B = (B[2] - B[0]) * (B[3] - B[1])
union = area_A + area_B - inter                  # subtract overlap, or it is counted twice

print("overlap area:", inter)
print("union area  :", union)
print("IoU         :", inter / union)

In [ ]:
def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_w = max(0, min(ax2, bx2) - max(ax1, bx1))
    inter_h = max(0, min(ay2, by2) - max(ay1, by1))
    inter = inter_w * inter_h
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / union if union > 0 else 0.0

def plot_boxes(a, b, title=""):
    fig, ax = plt.subplots(figsize=(4, 4))
    for box, colour, name in [(a, "tab:blue", "A (truth)"), (b, "tab:orange", "B (prediction)")]:
        ax.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                   fill=True, alpha=0.3, color=colour, label=name))
    ax.set_xlim(-20, 220); ax.set_ylim(220, -20)      # y grows downward, like in an image
    ax.set_aspect("equal"); ax.legend(loc="lower right"); ax.set_title(f"{title} IoU = {iou(a, b):.2f}")
    plt.show()

assert abs(iou(A, B) - 2500 / 17500) < 1e-9
plot_boxes(A, B)

## Experiment: slide the prediction away

How fast does IoU drop when the predicted box is only slightly off?

In [ ]:
truth = (0, 0, 100, 100)
shifts = list(range(0, 101, 10))
ious = [iou(truth, (s, 0, 100 + s, 100)) for s in shifts]

for s, v in zip(shifts, ious):
    print(f"shift {s:3d} px  ->  IoU {v:.2f}")

plt.plot(shifts, ious, "o-")
plt.axhline(0.5, color="red", linestyle="--", label="IoU = 0.5 threshold")
plt.xlabel("how far the predicted box is shifted (pixels)")
plt.ylabel("IoU")
plt.legend(); plt.grid(True); plt.show()

### What to observe

Shifting a 100-pixel box by only **33 pixels** already drops IoU to about 0.5. IoU is strict: a box that *looks* roughly right can still fail the 0.5 rule.

### Challenge 2

1. Keep the shift at 0 but make the prediction **twice as wide** `(0, 0, 200, 100)`. What is the IoU? Calculate by hand first.
2. Make a prediction fully **inside** the truth box. Why is IoU not 1?

---
# Part 4 — Confidence, true positives, false positives, false negatives

## Concept

```text
                     Model says "hand"        Model says "no hand"
 There IS a hand       True Positive (TP)       False Negative (FN)  ← missed it
 There is NO hand      False Positive (FP)      True Negative (TN)
```

For boxes we also need the box to be right: a hand detected with IoU < 0.5 counts as a **false positive** *and* the real hand is a **false negative**.

The model attaches a **confidence** to every detection. We choose a **threshold**: keep only detections above it.

```text
precision = TP / (TP + FP)    "of the hands I reported, how many were real?"
recall    = TP / (TP + FN)    "of the real hands, how many did I find?"
```

## Experiment: made-up detections

Each row is one detection: `[confidence, IoU with the true box]`. There are 7 real hands in total.

In [ ]:
detections = np.array([
    [0.95, 0.88], [0.91, 0.75], [0.85, 0.62], [0.80, 0.30], [0.72, 0.55],
    [0.60, 0.10], [0.55, 0.52], [0.40, 0.05], [0.30, 0.20],
])
n_real_hands = 7

rows = []
for threshold in [0.9, 0.7, 0.5, 0.3, 0.1]:
    kept = detections[:, 0] >= threshold
    tp = int(np.sum(kept & (detections[:, 1] >= 0.5)))
    fp = int(np.sum(kept & (detections[:, 1] < 0.5)))
    fn = n_real_hands - tp
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / n_real_hands
    rows.append((threshold, tp, fp, fn, precision, recall))
    print(f"threshold {threshold:.1f}:  TP={tp}  FP={fp}  FN={fn}   precision={precision:.2f}  recall={recall:.2f}")

th = [r[0] for r in rows]
plt.plot(th, [r[4] for r in rows], "o-", label="precision")
plt.plot(th, [r[5] for r in rows], "s-", label="recall")
plt.xlabel("confidence threshold"); plt.legend(); plt.grid(True); plt.show()

### What to observe

- **High threshold** → few detections, almost all correct (high precision) but many hands missed (low recall).
- **Low threshold** → catches more hands (high recall) but also junk (low precision).
- There is no free lunch: you choose where to sit on this trade-off. For a sign-language app, which mistake is worse: **showing a wrong sign** or **not showing anything**?

## Now with the real model

`min_hand_detection_confidence` is MediaPipe's version of this threshold. Run the detector at different values. If you have a photo **without** a hand, save it as `data/no_hand.jpg` to see false positives or true negatives.

In [ ]:
test_photos = [DATA / "myhand.jpg", DATA / "no_hand.jpg"]

for thr in [0.1, 0.5, 0.9]:
    det = create_detector(min_detection=thr)
    for p in test_photos:
        if not p.exists():
            continue
        image = cv2.imread(str(p))
        res = detect_hand(det, image)
        n = len(res.hand_landmarks)
        score = round(res.handedness[0][0].score, 3) if n else None
        print(f"threshold {thr}  {p.name:12s}  hands found: {n}  score: {score}")

### What to observe

Does lowering the threshold ever make the model "see" a hand in the no-hand photo? If not, MediaPipe is quite reliable on easy photos. Try a harder one (a face, a glove, a cartoon hand, a hand in the dark).

---
# Part 5 — Compare with a human label

Ground truth is a box **a human drew**. Look at your photo with a pixel grid, read off the corners of your hand, and enter them below.

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.grid(True, alpha=0.4)
plt.title("Read the x (horizontal) and y (vertical) pixel values from the axes")
plt.show()

In [ ]:
# Fill in YOUR box after looking at the grid above: (x1, y1, x2, y2)
gt_box = None            # e.g. gt_box = (120, 80, 540, 610)

if gt_box is None:
    print("Set gt_box above, then run this cell again.")
else:
    pred_box = box_from_keypoints(points_px, pad=0, img_w=W, img_h=H)
    print("your box     :", gt_box)
    print("model's box  :", pred_box)
    print("IoU          :", round(iou(gt_box, pred_box), 3))

    check = img.copy()
    cv2.rectangle(check, gt_box[:2], gt_box[2:], (255, 0, 0), 3)      # blue = your label
    cv2.rectangle(check, pred_box[:2], pred_box[2:], (0, 255, 0), 3)  # green = model
    show_bgr(check, "blue = human label, green = model box")

### What to observe

Your IoU is probably **not 1.0** even though the model is right. Whose box is "true"? A human's box is not exact either. Real datasets have label noise too, which is important when we train models later.

---
# 9. Results

```text
Image ─► MediaPipe ─► 21 keypoints ─► box (min/max + padding) ─► crop
                                       │
                                       └─► IoU against a human-drawn box
```

# 10. What did we learn?

1. A detection = **box + confidence**.
2. A box has four numbers `(x1, y1, x2, y2)`. Crop with `img[y1:y2, x1:x2]` (rows first!).
3. **IoU** = overlap / union. Small shifts reduce it a lot.
4. **Threshold** trades precision against recall: TP, FP, FN.
5. A box tells you *where* the hand is. Keypoints will tell you the *shape*.

# 11. Questions to think about

1. Why does `union = area_A + area_B - overlap` subtract the overlap?
2. If your system must **never** show a wrong sign, would you set the threshold high or low? What do you lose?
3. A hand is partly outside the image. What happens to the box? To the keypoints?
4. Why is a box not enough to recognise the letter "A" vs "B" in sign language?

# 12. Mini challenge

Write `evaluate_detection(gt_box, pred_box, iou_threshold=0.5)` that returns the string `"TP"` if IoU ≥ threshold, otherwise `"FP+FN"`. Test it on three pairs of boxes you invent.

---
## Experiment log

```text
EXP-001
Notebook : 04_hand_detection
Change   : min_hand_detection_confidence  0.1 / 0.5 / 0.9
Photos   : myhand.jpg, no_hand.jpg (+ your hard cases)
Observed :
Conclusion :
```

**Next:** `05_hand_keypoints.ipynb`: draw the skeleton, understand the 21 numbers, normalize them, go live with the webcam, and train your own keypoint model.